# Attention in the encoder–decoder
A context vector by hand, then two encoder–decoders trained on the same English–French pairs, without and
with (Bahdanau) attention, scored by sentence length, and the attention weights of one real sentence.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
import re, time, random, collections
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

np.set_printoptions(precision=4, suppress=True)
print(tf.config.list_physical_devices("GPU"))

I0000 00:00:1791077323.787459 1110637 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## One context vector by hand
Three encoder hidden states of size 4, and alignment scores e for one decoder step. Softmax turns the
scores into weights that are positive and sum to 1; the context vector is the weighted sum of the states.

In [2]:
h = np.array([[1.0, 0.5, 0.6, 0.3],
              [0.2, 0.9, 0.1, 0.4],
              [0.7, 0.1, 0.8, 0.5]])
e = np.array([0.5, 0.2, 1.8])
alpha = np.exp(e) / np.exp(e).sum()
c = alpha @ h
print("weights:", alpha.round(3), " sum:", alpha.sum())
print("context vector:", c.round(3))

weights: [0.185 0.137 0.678]  sum: 1.0
context vector: [0.687 0.283 0.667 0.449]


## The data
English–French pairs (the Keras example dataset), grouped by their English side so that test sentences
never appear in training. English sentences of up to 16 words. Long sentences are rare, so the training
set keeps every pair with 9 or more English words and fills up to 60,000 pairs with shorter ones.

In [3]:
path = keras.utils.get_file("fra-eng.zip", origin="http://storage.googleapis.com/download.tensorflow.org/data/fra-eng.zip",
                            extract=True)
lines = open(os.path.join(path, "fra.txt"), encoding="utf8").read().strip().split("\n")
tok = lambda s: re.findall(r"[\w']+|[^\w\s]", s.lower())
groups = collections.defaultdict(list)
for line in lines:
    e_, f_ = line.split("\t")[:2]
    e_, f_ = tok(e_), tok(f_)
    if len(e_) <= 16 and len(f_) <= 20:
        groups[tuple(e_)].append(f_)
BUCKETS = [(1, 4), (5, 7), (8, 10), (11, 13), (14, 16)]
bucket = lambda n: next(i for i, (a, b) in enumerate(BUCKETS) if a <= n <= b)
keys = sorted(groups)
random.Random(0).shuffle(keys)
test_keys, rest = [], []
taken = collections.Counter()
for k in keys:
    b = bucket(len(k))
    if taken[b] < 200:
        test_keys.append(k); taken[b] += 1
    else:
        rest.append(k)
pairs = [(list(k), f) for k in rest for f in groups[k]]
long_ = [p for p in pairs if len(p[0]) >= 9]
short = [p for p in pairs if len(p[0]) < 9]
random.Random(1).shuffle(short)
train = long_ + short[:60000 - len(long_)]
random.Random(2).shuffle(train)
print(len(train), "training pairs,", len(long_), "of them with 9+ English words;", len(test_keys), "test sentences",
      dict(sorted(taken.items())))

60000 training pairs, 39070 of them with 9+ English words; 1000 test sentences {0: 200, 1: 200, 2: 200, 3: 200, 4: 200}


In [4]:
def vocab(seqs, n):
    cnt = collections.Counter(w for s in seqs for w in s)
    words = ["<pad>", "<unk>", "<start>", "<end>"] + [w for w, _ in cnt.most_common(n)]
    return {w: i for i, w in enumerate(words)}, words

en_ix, en_w = vocab([e_ for e_, _ in train], 8000)
fr_ix, fr_w = vocab([f_ for _, f_ in train], 10000)
LE, LF = 16, 22

def encode(seqs, ix, L, add=False):
    X = np.zeros((len(seqs), L), "int32")
    for i, s in enumerate(seqs):
        s = ["<start>"] + s + ["<end>"] if add else s
        ids = [ix.get(w, 1) for w in s][:L]
        X[i, :len(ids)] = ids
    return X

Xe = encode([e_ for e_, _ in train], en_ix, LE)
Yf = encode([f_ for _, f_ in train], fr_ix, LF, add=True)
Xd, Yd = Yf[:, :-1], Yf[:, 1:]
test_en = [list(k) for k in test_keys]
test_refs = [groups[k] for k in test_keys]
Xe_test = encode(test_en, en_ix, LE)
print(len(en_w), len(fr_w), Xe.shape, Xd.shape)

8004 10004 (60000, 16) (60000, 21)


## The two models, written as one loop over the decoder steps
Encoder: embedding + bidirectional LSTM (128 units each way, as in Bahdanau et al. 2015, section 3.2), keeping
every hidden state h_1..h_n (forward and backward joined: 256 numbers). Decoder: an LSTM cell, one step per French
word. Without attention the decoder starts from the encoder's final states and sees nothing else.
With attention (Bahdanau et al. 2015), at step i:
e_ij = v^T tanh(W s_{i-1} + U h_j), alpha_i = softmax over j (padding excluded), c_i = sum_j alpha_ij h_j,
and c_i joins the previous word as the LSTM cell's input and the new state as the softmax layer's input.

In [5]:
D, H, A = 128, 256, 128

class Seq2Seq(keras.Model):
    def __init__(self, attention, seed):
        super().__init__()
        keras.utils.set_random_seed(seed)
        self.attention = attention
        self.emb_e = keras.layers.Embedding(len(en_w), D)
        self.enc = keras.layers.Bidirectional(keras.layers.LSTM(H // 2, return_sequences=True, return_state=True))
        self.emb_d = keras.layers.Embedding(len(fr_w), D)
        self.cell = keras.layers.LSTMCell(H)
        self.out = keras.layers.Dense(len(fr_w))
        if attention:
            self.W = keras.layers.Dense(A, use_bias=False)   # applied to s_{i-1}
            self.U = keras.layers.Dense(A)                   # applied to h_j
            self.v = keras.layers.Dense(1, use_bias=False)

    def call(self, xe, first, gold=None, steps=LF - 1):
        hs, hf, cf, hb, cb = self.enc(self.emb_e(xe))      # forward and backward final states
        h, c = tf.concat([hf, hb], -1), tf.concat([cf, cb], -1)
        pad = tf.cast(xe == 0, tf.float32)                   # 1 where the source is padding
        if self.attention:
            Uh = self.U(hs)                                  # does not depend on i: computed once
        word, logits, alphas = first, [], []
        for t in range(steps):
            x = self.emb_d(word)
            if self.attention:
                e = tf.squeeze(self.v(tf.tanh(self.W(h)[:, None, :] + Uh)), -1) - 1e9 * pad
                a = tf.nn.softmax(e, axis=-1)
                ctx = tf.reduce_sum(a[:, :, None] * hs, axis=1)
                alphas.append(a)
                x = tf.concat([x, ctx], -1)
            o, (h, c) = self.cell(x, [h, c])
            lg = self.out(tf.concat([o, ctx], -1) if self.attention else o)
            logits.append(lg)
            word = gold[:, t + 1] if (gold is not None and t + 1 < steps) else tf.argmax(lg, -1, output_type=tf.int32)
        return tf.stack(logits, 1), (tf.stack(alphas, 1) if alphas else None)

loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")

def ngrams(s, n):
    return collections.Counter(tuple(s[i:i + n]) for i in range(len(s) - n + 1))

def bleu(hyps, refss):
    """Corpus BLEU-4 with several references per sentence (Papineni et al. 2002), 0-100."""
    match, total = np.zeros(4), np.zeros(4)
    hyp_len = ref_len = 0
    for hy, refs in zip(hyps, refss):
        hyp_len += len(hy)
        ref_len += min((abs(len(r) - len(hy)), len(r)) for r in refs)[1]
        for n in range(1, 5):
            best = collections.Counter()
            for r in refs:
                best |= ngrams(r, n)
            match[n - 1] += sum(min(k, best[g]) for g, k in ngrams(hy, n).items())
            total[n - 1] += max(len(hy) - n + 1, 0)
    if match.min() == 0:
        return 0.0
    bp = 1.0 if hyp_len > ref_len else np.exp(1 - ref_len / max(hyp_len, 1))
    return 100 * bp * np.exp(np.mean(np.log(match / total)))

def greedy(m, X, batch=100):
    """Greedy decoding in batches (keeps GPU memory small); returns words and attention weights."""
    ids, als = [], []
    for k in range(0, len(X), batch):
        lg, al = m(X[k:k + batch], tf.fill([len(X[k:k + batch])], fr_ix["<start>"]))
        ids.append(tf.argmax(lg, -1).numpy())
        als.append(al.numpy() if al is not None else None)
    al = np.concatenate(als) if als[0] is not None else None
    out = []
    for row in np.concatenate(ids):
        s = []
        for k in row:
            if fr_w[k] == "<end>":
                break
            s.append(fr_w[k])
        out.append(s)
    return out, al

def train_model(attention, seed, epochs=12):
    m = Seq2Seq(attention, seed)
    opt = keras.optimizers.Adam()
    m(Xe[:2], Xd[:2, 0], Xd[:2])

    @tf.function
    def step(xe, xd, yd):
        with tf.GradientTape() as tape:
            lg, _ = m(xe, xd[:, 0], xd)
            mask = tf.cast(yd > 0, tf.float32)
            loss = tf.reduce_sum(loss_fn(yd, lg) * mask) / tf.reduce_sum(mask)
        opt.apply_gradients(zip(tape.gradient(loss, m.trainable_variables), m.trainable_variables))
        return loss

    ds = tf.data.Dataset.from_tensor_slices((Xe, Xd, Yd)).shuffle(len(Xe), seed=seed).batch(128)
    t0 = time.time()
    for ep in range(epochs):
        loss = np.mean([float(step(a, b, c)) for a, b, c in ds])
    print(f"attention={attention} seed={seed}: final training loss {loss:.3f}, {time.time() - t0:.0f} s,",
          f"{m.count_params():,} parameters")
    return m

## Training: 12 epochs each, 3 seeds, then BLEU on the test sentences by English length

In [6]:
rows, models = [], {}
for attention in (False, True):
    for seed in range(3):
        m = train_model(attention, seed)
        models[(attention, seed)] = m
        hyp, _ = greedy(m, Xe_test)
        for b, (lo, hi) in enumerate(BUCKETS):
            idx = [i for i, k in enumerate(test_keys) if lo <= len(k) <= hi]
            rows.append(dict(attention=attention, seed=seed, words=f"{lo}-{hi}",
                             bleu=bleu([hyp[i] for i in idx], [test_refs[i] for i in idx])))
        rows.append(dict(attention=attention, seed=seed, words="all", bleu=bleu(hyp, test_refs)))
res = pd.DataFrame(rows)
res.to_csv("data/bleu_by_length.csv", index=False)
print(res.pivot_table(index="words", columns="attention", values="bleu", aggfunc="mean").round(1))

attention=False seed=0: final training loss 1.936, 260 s, 5,533,460 parameters


attention=False seed=1: final training loss 1.904, 240 s, 5,533,460 parameters


attention=False seed=2: final training loss 1.808, 220 s, 5,533,460 parameters


attention=True seed=0: final training loss 0.755, 553 s, 8,422,420 parameters


attention=True seed=1: final training loss 0.790, 597 s, 8,422,420 parameters


attention=True seed=2: final training loss 0.754, 422 s, 8,422,420 parameters


attention  False  True 
words                  
1-4         14.3   33.3
11-13        8.3   24.3
14-16        5.9   17.5
5-7         13.9   30.4
8-10        13.7   35.9
all          9.8   25.7


## The attention weights of one real sentence
A long test sentence translated by the first attention model; each row of the weights is one French word,
each column one English word.

In [7]:
m = models[(True, 0)]
cands = [i for i, k in enumerate(test_keys) if 11 <= len(k) <= 13]
hyps, al = greedy(m, Xe_test[cands])
scores = [bleu([hy], [test_refs[i]]) for hy, i in zip(hyps, cands)]
for j in np.argsort(scores)[::-1][:8]:
    print(round(scores[j], 1), " ".join(test_en[cands[j]]), "->", " ".join(hyps[j]))

100.0 she advised him to talk about his life in america . -> elle lui conseilla de parler de sa vie en amérique .
100.0 i'd really like to know why he did that sort of thing . -> j'aimerais vraiment savoir pourquoi il a fait ce genre de chose .
100.0 i just think you should be more careful , that's all . -> je pense simplement que vous devriez être plus prudente , c'est tout .
100.0 instead of going to school , he stayed at home . -> au lieu d'aller à l'école , il est resté à la maison .
80.7 as far as i know , she is a very good person . -> pour autant que je sache , c'est une bonne idée .
80.0 is there something in particular that you want to study ? -> y a - t - il quelque chose de particulier que tu veuilles étudier ?
78.3 he has a bookstore in the center of the city . -> il a une librairie dans le centre - ville .
75.1 it seems to me that she is a little selfish . -> il me semble qu'elle est un peu <unk> .


In [8]:
pick = int(np.argsort(scores)[::-1][0])
src, hyp = test_en[cands[pick]], hyps[pick]
W = al[pick][:len(hyp) + 1, :len(src)]
print(" ".join(src), "->", " ".join(hyp))
print(pd.DataFrame(W.round(2), index=hyp + ["<end>"], columns=src))
pd.DataFrame(W, index=hyp + ["<end>"], columns=src).to_csv("data/attention_weights.csv")
print("number of weights for this pair:", W.shape[0], "x", W.shape[1], "=", W.size)

she advised him to talk about his life in america . -> elle lui conseilla de parler de sa vie en amérique .
            she  advised   him    to  talk  about  his  life    in  america  \
elle       0.01     0.14  0.62  0.23  0.00   0.00  0.0  0.00  0.00      0.0   
lui        0.00     0.00  0.88  0.12  0.00   0.00  0.0  0.00  0.00      0.0   
conseilla  0.00     0.00  0.03  0.93  0.01   0.03  0.0  0.00  0.00      0.0   
de         0.00     0.00  0.03  0.81  0.09   0.06  0.0  0.00  0.01      0.0   
parler     0.00     0.00  0.00  0.01  0.01   0.88  0.0  0.01  0.06      0.0   
de         0.00     0.00  0.00  0.00  0.00   0.14  0.0  0.01  0.79      0.0   
sa         0.00     0.00  0.00  0.00  0.00   0.01  0.0  0.10  0.81      0.0   
vie        0.00     0.00  0.00  0.00  0.00   0.00  0.0  0.03  0.89      0.0   
en         0.00     0.00  0.00  0.00  0.00   0.00  0.0  0.00  0.03      0.0   
amérique   0.00     0.00  0.00  0.00  0.00   0.00  0.0  0.00  0.01      0.0   
.          0.00     0.0